# 40 — Agent Observability

**What you'll learn:**
- How `BaseCallbackHandler` hooks into the LangChain/LangGraph execution lifecycle
- What `run_id` is and why it correlates `on_llm_start` with `on_llm_end`
- How to measure per-call latency using `time.time()` in `on_llm_start` / `on_llm_end`
- How to extract token counts from `response.llm_output["token_usage"]`
- How to handle errors via `on_llm_error` without crashing the agent

**vs LangSmith:** LangSmith is a hosted tracing platform that requires an account and API key. This notebook shows the *zero-dependency* alternative — a plain Python class that attaches directly to `invoke()` calls and produces the same core telemetry locally, with no external service needed.

In [ ]:
# Colab install guard
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    %pip install -q langchain langchain-openai langgraph python-dotenv

print(f"Running in Colab: {IN_COLAB}")

In [ ]:
# API key setup
import os

if IN_COLAB:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
else:
    from dotenv import load_dotenv
    load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "Set OPENAI_API_KEY before continuing"
print("API key loaded.")

In [ ]:
# Callback lifecycle explanation
#
# LangChain fires these events on every LLM call:
#
#   on_llm_start(serialized, prompts, run_id, ...)
#     - Called just before the HTTP request is sent to the model
#     - run_id is a UUID that uniquely identifies THIS call
#     - We store time.time() keyed by run_id to start the clock
#
#   on_llm_end(response, run_id, ...)
#     - Called after the model returns a response
#     - Same run_id as the matching on_llm_start
#     - response.llm_output["token_usage"] holds prompt + completion counts
#     - We pop the start time, compute elapsed ms, and append to self.calls
#
#   on_llm_error(error, run_id, ...)
#     - Called if the LLM raises an exception (rate limit, network error, etc.)
#     - We record the error string so report() can count failures
#
# The callback is STATEFUL -- it accumulates all calls in self.calls
# and exposes report() for a summary across all calls in the session.

print("Callback lifecycle: start -> end/error")

In [ ]:
import time
from typing import Any, TypedDict

from langchain.callbacks.base import BaseCallbackHandler
from langchain_core.messages import HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.graph import END, StateGraph


class ObservabilityCallback(BaseCallbackHandler):
    def __init__(self):
        self.calls: list[dict] = []
        self._starts: dict[str, float] = {}

    def on_llm_start(self, serialized: dict, prompts: list, run_id: Any, **kwargs):
        self._starts[str(run_id)] = time.time()

    def on_llm_end(self, response: Any, run_id: Any, **kwargs):
        elapsed = round((time.time() - self._starts.pop(str(run_id), time.time())) * 1000)
        usage = {}
        if response.llm_output:
            usage = response.llm_output.get("token_usage", {})
        self.calls.append({
            "latency_ms": elapsed,
            "prompt_tokens": usage.get("prompt_tokens", 0),
            "completion_tokens": usage.get("completion_tokens", 0),
        })

    def on_llm_error(self, error: Any, run_id: Any, **kwargs):
        self.calls.append({"error": str(error)})

    def report(self) -> dict:
        ok = [c for c in self.calls if "error" not in c]
        return {
            "total_calls": len(self.calls),
            "errors": len(self.calls) - len(ok),
            "avg_latency_ms": round(sum(c["latency_ms"] for c in ok) / max(len(ok), 1)),
            "total_tokens": sum(c.get("prompt_tokens", 0) + c.get("completion_tokens", 0) for c in ok),
        }


class ObservabilityState(TypedDict):
    task: str
    response: str


def create_workflow(callback=None):
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.3)
    graph = StateGraph(ObservabilityState)

    def node(state):
        result = llm.invoke(
            [HumanMessage(content=state["task"])],
            config={"callbacks": [callback]} if callback else {},
        )
        return {"response": result.content}

    graph.add_node("llm", node)
    graph.set_entry_point("llm")
    graph.add_edge("llm", END)
    return graph.compile()


print("Graph compiled.")

In [ ]:
DEMO_TASKS = [
    "Explain what an LLM callback handler does in one sentence.",
    "What are three benefits of agent observability?",
]

cb = ObservabilityCallback()
app = create_workflow(callback=cb)

for task in DEMO_TASKS:
    print(f"\nTASK: {task}")
    result = app.invoke({"task": task, "response": ""})
    print(f"Response: {result['response'][:120]}")

print("\n--- Telemetry Report ---")
report = cb.report()
for k, v in report.items():
    print(f"  {k}: {v}")

## Exercises

1. **Chain timing:** Add `on_chain_start` and `on_chain_end` methods to `ObservabilityCallback` to also track the time spent in each LangGraph node (chain), not just LLM calls.

2. **File logger:** Write a subclass of `ObservabilityCallback` that appends each call record as a JSON line to a log file, so telemetry persists across sessions.

3. **Matplotlib dashboard:** After running `DEMO_TASKS`, use `cb.calls` and `matplotlib` to plot a bar chart of latency per call and a pie chart of token distribution (prompt vs completion).

4. **Overhead measurement:** Run the same task once with `create_workflow(callback=cb)` and once with `create_workflow(callback=None)`. Compare the two latencies — how much overhead does the callback add?

---

### What's next?

- **[29-llm-judge-harness](../29-llm-judge-harness/)** — use an LLM to evaluate agent output quality (complements observability with semantic scoring)
- **[16-rag-eval-notebook](../16-rag-eval-notebook/)** — RAGAS evaluation metrics for RAG pipelines (faithfulness, answer relevancy, context precision)